# E2E Finance FAQ Non-Instruction Fine-Tuning (Colab)
This notebook sets up the repo on Colab, prepares data, builds token blocks, applies LoRA/QLoRA, and runs non-instruction fine-tuning.

In [1]:
# 1) Runtime check
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

CUDA available: False


In [2]:
# 2) Set input corpus path (no Google Drive required)
from pathlib import Path

# Put your corpus at this exact path in Colab.
# Example accepted formats: CSV with complaint narratives.
INPUT_CORPUS_PATH = Path('/content/complaints-2026-07-04_02_07.csv')

# Optional: fallback auto-discovery pattern in /content.
INPUT_CORPUS_GLOB = 'complaints-*.csv'

print('INPUT_CORPUS_PATH:', INPUT_CORPUS_PATH)

INPUT_CORPUS_PATH: /content/complaints-2026-07-04_02_07.csv


In [3]:
# 3) Clone or refresh repository
import os
REPO_URL = 'https://github.com/Swapperss/FinanceQAAssistance.git'
PROJECT_ROOT = '/content/FinanceQAAssistance'
if not os.path.exists(PROJECT_ROOT):
    !git clone {REPO_URL} {PROJECT_ROOT}
else:
    print('Repository already exists, pulling latest changes...')
    %cd {PROJECT_ROOT}
    !git pull
%cd {PROJECT_ROOT}
!git branch --show-current

Cloning into '/content/FinanceQAAssistance'...
remote: Enumerating objects: 45, done.
remote: Counting objects: 100% (45/45), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 45 (delta 7), reused 39 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (45/45), 4.33 MiB | 6.42 MiB/s, done.
Resolving deltas: 100% (7/7), done.
/content/FinanceQAAssistance
main


In [4]:
# 4) Install dependencies
!pip -q install -U pip
!pip -q install -r requirements.txt
!pip -q install -U transformers accelerate peft bitsandbytes datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 30.2 MB/s eta 0:00:00


In [ ]:
# 5) Load project config and logger
import sys
from pathlib import Path
project_root = Path('/content/FinanceQAAssistance').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
import importlib.util
config_spec = importlib.util.spec_from_file_location('finance_config', project_root / 'config' / 'config.py')
config_module = importlib.util.module_from_spec(config_spec)
assert config_spec.loader is not None
config_spec.loader.exec_module(config_module)
Config = config_module.Config
from utils.logger import get_logger
config = Config()
logger = get_logger('non_instruction_colab')
logger.info('Colab E2E notebook started')
print('Loaded config model:', config.model_name)

2026-07-12 13:12:12,068 | INFO | non_instruction_colab | Colab E2E notebook started
INFO:non_instruction_colab:Colab E2E notebook started


Loaded config model: Qwen/Qwen2.5-0.5B


In [6]:
# 6) Create required folders
raw_dir = project_root / config.raw_data_dir
non_instruction_dir = project_root / config.non_instruction_dir
artifacts_output_dir = project_root / config.output_dir
artifacts_adapter_dir = project_root / config.adapter_dir
for p in [raw_dir, non_instruction_dir, artifacts_output_dir, artifacts_adapter_dir, project_root / 'logs']:
    p.mkdir(parents=True, exist_ok=True)
print('raw_dir:', raw_dir)
print('non_instruction_dir:', non_instruction_dir)
print('output_dir:', artifacts_output_dir)
print('adapter_dir:', artifacts_adapter_dir)

raw_dir: /content/FinanceQAAssistance/data/raw
non_instruction_dir: /content/FinanceQAAssistance/data/non_instruction_dataset
output_dir: /content/FinanceQAAssistance/artifacts/non_instruction_output
adapter_dir: /content/FinanceQAAssistance/artifacts/non_instruction_adapter


In [7]:
# 7) Resolve input corpus and place it in project data/raw
import shutil
from pathlib import Path

target_csv_path = raw_dir / config.preferred_raw_csv_filename

if INPUT_CORPUS_PATH.exists():
    shutil.copy2(INPUT_CORPUS_PATH, target_csv_path)
    print('Copied input corpus to:', target_csv_path)
else:
    # Auto-discover likely corpus files in /content if INPUT_CORPUS_PATH is not set/found.
    matches = sorted(Path('/content').glob(INPUT_CORPUS_GLOB))
    if matches:
        source_path = matches[-1]
        shutil.copy2(source_path, target_csv_path)
        print('Auto-discovered and copied corpus:', source_path)
        print('Copied to:', target_csv_path)
    else:
        raise FileNotFoundError(
            'No input corpus found. Upload a CSV to /content/complaints-2026-07-04_02_07.csv '
            f'or place a file matching {INPUT_CORPUS_GLOB} under /content.'
        )

print('CSV exists in raw_dir:', target_csv_path.exists())

Copied input corpus to: /content/FinanceQAAssistance/data/raw/complaints-2026-07-04_02_07.csv
CSV exists in raw_dir: True


In [8]:
# 8) Extract raw text from complaints CSV
import pandas as pd
csv_path = raw_dir / config.preferred_raw_csv_filename
if not csv_path.exists():
    matches = sorted(raw_dir.glob(config.raw_csv_pattern))
    if not matches:
        raise FileNotFoundError(f'No complaints CSV found in {raw_dir}')
    csv_path = matches[-1]
out_path = non_instruction_dir / config.raw_text_filename
df = pd.read_csv(csv_path, dtype=str, low_memory=False)
text_col = config.source_text_column
if text_col not in df.columns:
    raise ValueError(f'Column not found: {text_col}')
texts = df[text_col].dropna().astype(str).str.strip()
texts = texts[texts.str.len() > config.min_chars_per_paragraph]
out_path.write_text('\n\n'.join(texts.tolist()), encoding='utf-8')
logger.info('Raw extraction complete')
logger.info('Input rows: %s', len(df))
logger.info('Extracted text rows: %s', len(texts))
logger.info('Saved file: %s', out_path.resolve())
print('Extracted rows:', len(texts))
print('Saved to:', out_path.resolve())

2026-07-12 13:13:18,793 | INFO | non_instruction_colab | Raw extraction complete
INFO:non_instruction_colab:Raw extraction complete
2026-07-12 13:13:18,797 | INFO | non_instruction_colab | Input rows: 20859
INFO:non_instruction_colab:Input rows: 20859
2026-07-12 13:13:18,800 | INFO | non_instruction_colab | Extracted text rows: 4702
INFO:non_instruction_colab:Extracted text rows: 4702
2026-07-12 13:13:18,802 | INFO | non_instruction_colab | Saved file: /content/FinanceQAAssistance/data/non_instruction_dataset/raw_extracted_text.txt
INFO:non_instruction_colab:Saved file: /content/FinanceQAAssistance/data/non_instruction_dataset/raw_extracted_text.txt


Extracted rows: 4702
Saved to: /content/FinanceQAAssistance/data/non_instruction_dataset/raw_extracted_text.txt


In [9]:
# 9) Clean text and create final non-instruction corpus
import re
in_path = non_instruction_dir / config.raw_text_filename
out_path = non_instruction_dir / config.non_instruction_filename
raw_text = in_path.read_text(encoding='utf-8')
def clean_text(text: str) -> str:
    text = re.sub(r'\bX{2,}\b', ' ', text)
    text = re.sub(r'https?://\S+|www\.\S+', ' [URL] ', text)
    text = re.sub(r'\b[\w\.-]+@[\w\.-]+\.\w+\b', ' [EMAIL] ', text)
    text = re.sub(r'\b\d{5,}\b', ' [NUMBER] ', text)
    text = re.sub(r'[^\w\s\.,!?;:"()\-\/]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text
paragraphs = [p.strip() for p in raw_text.split('\n\n') if p.strip()]
cleaned = [clean_text(p) for p in paragraphs]
cleaned = [p for p in cleaned if len(p) >= config.min_chars_per_paragraph]
out_path.write_text('\n\n'.join(cleaned), encoding='utf-8')
logger.info('Cleaning complete')
logger.info('Input paragraphs: %s', len(paragraphs))
logger.info('Final non-instruction paragraphs: %s', len(cleaned))
logger.info('Saved final file: %s', out_path.resolve())
print('Final paragraphs:', len(cleaned))
print('Saved to:', out_path.resolve())

2026-07-12 13:13:33,757 | INFO | non_instruction_colab | Cleaning complete
INFO:non_instruction_colab:Cleaning complete
2026-07-12 13:13:33,762 | INFO | non_instruction_colab | Input paragraphs: 18824
INFO:non_instruction_colab:Input paragraphs: 18824
2026-07-12 13:13:33,766 | INFO | non_instruction_colab | Final non-instruction paragraphs: 17479
INFO:non_instruction_colab:Final non-instruction paragraphs: 17479
2026-07-12 13:13:33,773 | INFO | non_instruction_colab | Saved final file: /content/FinanceQAAssistance/data/non_instruction_dataset/non_instruction_data.txt
INFO:non_instruction_colab:Saved final file: /content/FinanceQAAssistance/data/non_instruction_dataset/non_instruction_data.txt


Final paragraphs: 17479
Saved to: /content/FinanceQAAssistance/data/non_instruction_dataset/non_instruction_data.txt


In [10]:
# 10) Create Hugging Face dataset
from datasets import Dataset, DatasetDict
input_path = non_instruction_dir / config.non_instruction_filename
dataset_dir = non_instruction_dir / config.hf_dataset_dirname
text = input_path.read_text(encoding='utf-8')
paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]
records = [{'text': p} for p in paragraphs]
full_ds = Dataset.from_list(records)
split_ds = full_ds.train_test_split(test_size=config.test_size, seed=config.seed)
hf_ds = DatasetDict({'train': split_ds['train'], 'validation': split_ds['test']})
dataset_dir.mkdir(parents=True, exist_ok=True)
hf_ds.save_to_disk(str(dataset_dir))
logger.info('HF dataset creation complete')
logger.info('Total rows: %s', len(full_ds))
logger.info('Train rows: %s', len(hf_ds['train']))
logger.info('Validation rows: %s', len(hf_ds['validation']))
print(hf_ds)

Saving the dataset (0/1 shards):   0%|          | 0/15731 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1748 [00:00<?, ? examples/s]

2026-07-12 13:13:48,118 | INFO | non_instruction_colab | HF dataset creation complete
INFO:non_instruction_colab:HF dataset creation complete
2026-07-12 13:13:48,123 | INFO | non_instruction_colab | Total rows: 17479
INFO:non_instruction_colab:Total rows: 17479
2026-07-12 13:13:48,129 | INFO | non_instruction_colab | Train rows: 15731
INFO:non_instruction_colab:Train rows: 15731
2026-07-12 13:13:48,132 | INFO | non_instruction_colab | Validation rows: 1748
INFO:non_instruction_colab:Validation rows: 1748


DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 15731
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 1748
    })
})


In [11]:
# 11) Load tokenizer
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(config.model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'
print('Tokenizer loaded:', config.model_name)
print('Vocab size:', len(tokenizer))

config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Tokenizer loaded: Qwen/Qwen2.5-0.5B
Vocab size: 151665


In [12]:
# 12) Tokenization and text packing
dataset = hf_ds
def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, max_length=config.block_size)
tokenized_datasets = dataset.map(
    tokenize_function,
    remove_columns=dataset['train'].column_names,
    desc='Tokenizing text corpus',
)
tokenized_datasets

Tokenizing text corpus:   0%|          | 0/15731 [00:00<?, ? examples/s]

Tokenizing text corpus:   0%|          | 0/1748 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 15731
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 1748
    })
})

In [13]:
# 13) Create fixed-size training blocks
def create_training_blocks(tokenized_examples):
    all_input_ids = []
    all_attention_masks = []
    for input_ids in tokenized_examples['input_ids']:
        all_input_ids.extend(input_ids)
    for attention_mask in tokenized_examples['attention_mask']:
        all_attention_masks.extend(attention_mask)
    total_tokens = len(all_input_ids)
    usable_tokens = (total_tokens // config.block_size) * config.block_size
    if usable_tokens == 0:
        return {'input_ids': [], 'attention_mask': [], 'labels': []}
    all_input_ids = all_input_ids[:usable_tokens]
    all_attention_masks = all_attention_masks[:usable_tokens]
    input_id_blocks = []
    attention_mask_blocks = []
    for start_index in range(0, usable_tokens, config.block_size):
        end_index = start_index + config.block_size
        input_id_blocks.append(all_input_ids[start_index:end_index])
        attention_mask_blocks.append(all_attention_masks[start_index:end_index])
    labels = [block.copy() for block in input_id_blocks]
    return {'input_ids': input_id_blocks, 'attention_mask': attention_mask_blocks, 'labels': labels}
final_dataset = tokenized_datasets.map(
    create_training_blocks,
    batched=True,
    desc=f'Creating fixed-size training blocks of {config.block_size} tokens',
)
print(final_dataset)
print(final_dataset['train'][0].keys())

Creating fixed-size training blocks of 512 tokens:   0%|          | 0/15731 [00:00<?, ? examples/s]

Creating fixed-size training blocks of 512 tokens:   0%|          | 0/1748 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2418
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 275
    })
})
dict_keys(['input_ids', 'attention_mask', 'labels'])


In [14]:
# 14) Check device availability
import torch
use_cuda = torch.cuda.is_available()
print('CUDA available:', use_cuda)
if use_cuda:
    print('GPU:', torch.cuda.get_device_name(0))

CUDA available: False


In [15]:
# 15) Load base model
import gc
from transformers import AutoModelForCausalLM
gc.collect()
if use_cuda:
    torch.cuda.empty_cache()
if use_cuda:
    from transformers import BitsAndBytesConfig
    from peft import prepare_model_for_kbit_training
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        quantization_config=quantization_config,
        device_map='auto',
        trust_remote_code=True,
    )
    base_model = prepare_model_for_kbit_training(base_model)
else:
    base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )
base_model.config.use_cache = False
print('Base model loaded successfully.')

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Base model loaded successfully.


In [18]:
# 16) Update torchao for compatibility with peft
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 21.8 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [19]:
# 16) Apply LoRA adapters
from peft import LoraConfig, TaskType, get_peft_model
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=config.lora_r,
    lora_alpha=config.lora_alpha,
    lora_dropout=config.lora_dropout,
    bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


In [21]:
import gc
from transformers import AutoModelForCausalLM
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

# 17) Train model
train_dataset = final_dataset['train']
eval_dataset = final_dataset['validation']

# Keep only one sample so each epoch is one training step (3 epochs => 3 steps).
train_dataset = train_dataset.select(range(min(1, len(train_dataset))))
eval_dataset = eval_dataset.select(range(min(1, len(eval_dataset))))

training_args = TrainingArguments(
    output_dir=str(project_root / config.output_dir),
    num_train_epochs=config.num_train_epochs,
    per_device_train_batch_size=config.per_device_train_batch_size,
    per_device_eval_batch_size=config.per_device_eval_batch_size,
    gradient_accumulation_steps=1,
    learning_rate=config.learning_rate,
    warmup_ratio=config.warmup_ratio,
    weight_decay=config.weight_decay,
    logging_steps=1,
    logging_first_step=True,
    eval_steps=1,
    save_steps=1,
    save_total_limit=config.save_total_limit,
    max_steps=-1,
    # evaluation_strategy='steps', # Removed as it caused a TypeError
    # save_strategy='steps',       # Removed as it caused a TypeError
    bf16=use_cuda,
    fp16=False,
    report_to='none',
)

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collator,
)

train_result = trainer.train()
print('Training completed.')

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
1,3.188613
2,3.184741
3,2.768531


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training completed.


In [22]:
for log in trainer.state.log_history:
    print(log)

{'loss': 3.1886134147644043, 'grad_norm': 1.9409428834915161, 'learning_rate': 0.0, 'epoch': 1.0, 'step': 1}
{'loss': 3.1847407817840576, 'grad_norm': 1.954687237739563, 'learning_rate': 0.0002, 'epoch': 2.0, 'step': 2}
{'loss': 2.768531322479248, 'grad_norm': 1.8587231636047363, 'learning_rate': 0.0001, 'epoch': 3.0, 'step': 3}
{'train_runtime': 77.8312, 'train_samples_per_second': 0.039, 'train_steps_per_second': 0.039, 'total_flos': 3379473285120.0, 'train_loss': 3.047295173009237, 'epoch': 3.0, 'step': 3}


In [27]:
# 19) Quick inference check after training
import torch
import re

model.eval()

def ask_model(question: str, max_new_tokens: int = 120) -> str:
    prompt = f'Question: {question}\nAnswer:'
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            #temperature=0.0,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    answer_text = generated_text[len(prompt):].strip()
    stop_pattern = r"(A single-select problem:|Pick your answer from:).*"
    answer_text = re.split(stop_pattern, answer_text, maxsplit=1)[0].strip()
    return answer_text

sample_questions = [
    'What should I do if my bank charged an incorrect fee?',
    'How can I dispute an unauthorized charge on my credit card?',
    'What can I do if my payment was posted late because of a bank error?',
    'How do I request a refund for an unfair interest charge?',
    'What should I do if my account was charged fees after a system transition?',
]

for index, sample_question in enumerate(sample_questions, start=1):
    answer = ask_model(sample_question)
    print(f'Q{index}:', sample_question)
    print(f'A{index}:', answer)
    print('-' * 80)

Q1: What should I do if my bank charged an incorrect fee?
A1: If you believe that your bank has charged you an incorrect fee, you can file a complaint with the Federal Trade Commission (FTC) or your local bank. The FTC can help you resolve the issue and prevent future charges.
--------------------------------------------------------------------------------
Q2: How can I dispute an unauthorized charge on my credit card?
A2: To dispute an unauthorized charge, you can contact your credit card company by calling 1-800-686-1234 or by visiting their website at www.annualcreditcard.com. You will need to provide your credit card number, expiration date, and CVV code. Once you have provided the necessary information, your credit card company will investigate the dispute and provide you with a response within 30 days.
A: Yes, you can dispute an unauthorized charge on your credit card by contacting your credit card company. You can do this by calling 1-8
------------------------------------------

In [10]:
# 20) Finance instruction fine-tuning and instruction-style inference - setup
import json
import re
from pathlib import Path
import sys
import pandas as pd
from datasets import load_dataset

current_path = Path.cwd().resolve()
project_root = None
for candidate_path in [current_path, *current_path.parents]:
    if (candidate_path / 'config' / 'config.py').exists():
        project_root = candidate_path
        break
if project_root is None:
    raise FileNotFoundError('Could not locate config/config.py from the current working directory.')
project_root = Path(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import importlib.util
config_spec = importlib.util.spec_from_file_location('finance_config', project_root / 'config' / 'config.py')
config_module = importlib.util.module_from_spec(config_spec)
assert config_spec.loader is not None
config_spec.loader.exec_module(config_module)
Config = config_module.Config
config = globals().get('config', Config())
raw_dir = project_root / config.raw_data_dir
instruction_dir = project_root / config.instruction_dir
instruction_output_dir = project_root / config.instruction_output_dir
instruction_adapter_dir = project_root / config.instruction_adapter_dir
instruction_hf_dir = instruction_dir / config.instruction_hf_dataset_dirname
instruction_dataset_path = instruction_dir / config.instruction_dataset_filename

for folder_path in [instruction_dir, instruction_output_dir, instruction_adapter_dir, instruction_hf_dir]:
    folder_path.mkdir(parents=True, exist_ok=True)

raw_csv_path = raw_dir / config.preferred_raw_csv_filename
if not raw_csv_path.exists():
    matches = sorted(raw_dir.glob(config.raw_csv_pattern))
    if not matches:
        raise FileNotFoundError(f'No complaints CSV found in {raw_dir}')
    raw_csv_path = matches[-1]

df = pd.read_csv(raw_csv_path, dtype=str, low_memory=False).fillna('')

def normalize_text(value):
    return re.sub(r'\s+', ' ', str(value)).strip()

def finance_answer_from_issue(issue_text: str) -> str:
    issue_text = issue_text.lower()

    if any(keyword in issue_text for keyword in ['unauthorized', 'not authorized', 'fraud', 'identity theft']):
        return 'Contact your bank or card issuer immediately, lock or replace the card if needed, dispute the charge in writing, and monitor the account for any further suspicious activity.'

    if any(keyword in issue_text for keyword in ['fees or interest', 'incorrect fees or interest', 'charged too much interest', 'annual fee', 'fee']):
        return 'Review the statement, contact the bank or card issuer, dispute the fee in writing, and request a reversal or correction if the charge looks incorrect.'

    if any(keyword in issue_text for keyword in ['payment', 'late', 'posted late', 'not posted']):
        return 'Provide proof of payment, ask the bank to review the posting timeline, and request removal of any late fees or interest caused by a bank-side delay.'

    if any(keyword in issue_text for keyword in ['system transition', 'migration']):
        return 'Ask for a manual review, explain the system transition issue, and request reversal of any fees or interest caused by the transition.'

    if any(keyword in issue_text for keyword in ['credit limit']):
        return 'Contact customer service, ask for the decision in writing, and request a review of the account history and credit profile.'

    return 'Contact the bank customer service team, explain the issue in writing, ask for a manual review, and keep copies of all statements and correspondence.'

def build_instruction_record(row):
    product = normalize_text(row.get('Product', ''))
    issue = normalize_text(row.get('Issue', ''))
    sub_issue = normalize_text(row.get('Sub-issue', ''))
    complaint_text = normalize_text(row.get(config.source_text_column, ''))

    if issue:
        instruction = f'What should I do if my complaint is about {issue.lower()}?'
    elif product:
        instruction = f'What should I do if I have a problem with my {product.lower()} account?'
    else:
        instruction = 'What should I do about this finance complaint?'

    input_parts = []
    if product:
        input_parts.append(f'Product: {product}')
    if issue:
        input_parts.append(f'Issue: {issue}')
    if sub_issue:
        input_parts.append(f'Sub-issue: {sub_issue}')
    if complaint_text:
        input_parts.append(f'Complaint: {complaint_text[:1200]}')

    input_text = '\n'.join(input_parts).strip()
    output_text = finance_answer_from_issue(f'{product} {issue} {sub_issue}')

    return {
        'instruction': instruction,
        'input': input_text,
        'output': output_text,
        'product': product,
        'issue': issue,
        'sub_issue': sub_issue,
    }

max_instruction_rows = min(400, len(df))
instruction_records = []
for _, row in df.head(max_instruction_rows).iterrows():
    record = build_instruction_record(row)
    if record['instruction'] and record['output']:
        instruction_records.append(record)

if not instruction_records:
    raise ValueError('No finance instruction records could be created from the CSV.')

with open(instruction_dataset_path, 'w', encoding='utf-8') as file_handle:
    for record in instruction_records:
        file_handle.write(json.dumps(record, ensure_ascii=False) + '\n')

print('Created instruction records:', len(instruction_records))
print('Instruction dataset saved to:', instruction_dataset_path)
print('Sample instruction record:')
print(instruction_records[0])

Created instruction records: 400
Instruction dataset saved to: D:\Swapnil\DSML_LLM\AILLM\Projects\FinanceFAQAssistant_FT\data\instruction_dataset\finance_instruction_dataset.jsonl
Sample instruction record:
{'instruction': 'What should I do if my complaint is about other features, terms, or problems?', 'input': 'Product: Credit card\nIssue: Other features, terms, or problems\nSub-issue: Problem with customer service', 'output': 'Contact the bank customer service team, explain the issue in writing, ask for a manual review, and keep copies of all statements and correspondence.', 'product': 'Credit card', 'issue': 'Other features, terms, or problems', 'sub_issue': 'Problem with customer service'}


In [ ]:
# 21) Load and prepare the finance instruction dataset
from datasets import Dataset, DatasetDict, load_dataset
from transformers import AutoTokenizer

instruction_dataset = load_dataset('json', data_files=str(instruction_dataset_path), split='train')
tokenizer = AutoTokenizer.from_pretrained(config.model_name, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_instruction_record(record):
    instruction = str(record.get('instruction', '')).strip()
    input_text = str(record.get('input', '')).strip()
    output_text = str(record.get('output', '')).strip()

    if input_text:
        text = (
            f'### Instruction:\n{instruction}\n\n'
            f'### Input:\n{input_text}\n\n'
            f'### Response:\n{output_text}'
        )
    else:
        text = (
            f'### Instruction:\n{instruction}\n\n'
            f'### Response:\n{output_text}'
        )

    return {'text': text}

instruction_dataset = instruction_dataset.map(format_instruction_record)
instruction_datasets = instruction_dataset.train_test_split(test_size=config.test_size, seed=config.seed)
instruction_datasets['validation'] = instruction_datasets.pop('test')

print(instruction_datasets)
print('Train examples:', len(instruction_datasets['train']))
print('Validation examples:', len(instruction_datasets['validation']))

response_prefix_ids = tokenizer('### Response:\n', add_special_tokens=False)['input_ids']

def find_subsequence(sequence, pattern):
    if not pattern:
        return -1
    last_start = len(sequence) - len(pattern)
    for start_index in range(last_start + 1):
        if sequence[start_index:start_index + len(pattern)] == pattern:
            return start_index
    return -1

def tokenize_instruction_function(examples):
    tokens = tokenizer(
        examples['text'],
        truncation=True,
        padding='max_length',
        max_length=config.block_size,
    )

    labels = []
    for input_ids, attention_mask in zip(tokens['input_ids'], tokens['attention_mask']):
        row_labels = list(input_ids)

        # Ignore padded positions in loss for all modes.
        for idx, mask_value in enumerate(attention_mask):
            if mask_value == 0:
                row_labels[idx] = -100

        if config.instruction_train_response_only:
            response_start = find_subsequence(input_ids, response_prefix_ids)
            if response_start >= 0:
                for idx in range(response_start):
                    row_labels[idx] = -100

        labels.append(row_labels)

    tokens['labels'] = labels
    return tokens

instruction_tokenized_datasets = instruction_datasets.map(
    tokenize_instruction_function,
    batched=True,
    remove_columns=instruction_datasets['train'].column_names,
    desc='Tokenizing instruction dataset',
)

# POC speed mode: keep epochs=3 but reduce dataset size for faster turnaround.
poc_train_rows = min(config.instruction_poc_train_rows, len(instruction_tokenized_datasets['train']))
poc_val_rows = min(config.instruction_poc_validation_rows, len(instruction_tokenized_datasets['validation']))
instruction_tokenized_datasets['train'] = instruction_tokenized_datasets['train'].select(range(poc_train_rows))
instruction_tokenized_datasets['validation'] = instruction_tokenized_datasets['validation'].select(range(poc_val_rows))

print('POC train rows:', len(instruction_tokenized_datasets['train']))
print('POC validation rows:', len(instruction_tokenized_datasets['validation']))
print(instruction_tokenized_datasets)

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'product', 'issue', 'sub_issue', 'text'],
        num_rows: 360
    })
    validation: Dataset({
        features: ['instruction', 'input', 'output', 'product', 'issue', 'sub_issue', 'text'],
        num_rows: 40
    })
})
Train examples: 360
Validation examples: 40


Tokenizing instruction dataset: 100%|██████████| 40/40 [00:00<00:00, 791.70 examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 360
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 40
    })
})


In [ ]:
# 22) Load model and prepare instruction trainer
import gc
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

use_cuda = torch.cuda.is_available()
print('CUDA available:', use_cuda)
if use_cuda:
    print('GPU:', torch.cuda.get_device_name(0))

gc.collect()
if use_cuda:
    torch.cuda.empty_cache()

if use_cuda:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    instruction_base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        quantization_config=quantization_config,
        device_map='auto',
        trust_remote_code=True,
    )
    instruction_base_model = prepare_model_for_kbit_training(instruction_base_model)
else:
    instruction_base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

instruction_base_model.config.use_cache = False

instruction_lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=config.lora_r,
    lora_alpha=config.lora_alpha,
    lora_dropout=config.lora_dropout,
    bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)

instruction_model = get_peft_model(instruction_base_model, instruction_lora_config)
instruction_model.print_trainable_parameters()

instruction_data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Keep epochs=3 for requirement; use config-driven instruction-stage parameters.
instruction_training_args = TrainingArguments(
    output_dir=str(instruction_output_dir),
    num_train_epochs=config.instruction_num_train_epochs,
    max_steps=config.instruction_max_steps,
    per_device_train_batch_size=config.instruction_per_device_train_batch_size,
    per_device_eval_batch_size=config.instruction_per_device_eval_batch_size,
    gradient_accumulation_steps=config.instruction_gradient_accumulation_steps,
    learning_rate=config.instruction_learning_rate,
    warmup_ratio=config.instruction_warmup_ratio,
    weight_decay=config.instruction_weight_decay,
    logging_steps=config.instruction_logging_steps,
    logging_first_step=True,
    save_steps=config.instruction_save_steps,
    save_total_limit=config.instruction_save_total_limit,
    eval_strategy='no',
    fp16=use_cuda,
    bf16=False,
    report_to='none',
    remove_unused_columns=False,
)

instruction_trainer = Trainer(
    model=instruction_model,
    args=instruction_training_args,
    train_dataset=instruction_tokenized_datasets['train'],
    eval_dataset=instruction_tokenized_datasets['validation'],
    data_collator=instruction_data_collator,
)

print('Instruction Trainer is ready.')

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


CUDA available: False


Loading weights: 100%|██████████| 290/290 [00:04<00:00, 59.40it/s]
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497
Instruction Trainer is ready.


In [ ]:
# 23) Train and save the finance instruction adapter
instruction_train_result = instruction_trainer.train()
print('Instruction fine-tuning completed.')
print(instruction_train_result)

instruction_trainer.model.save_pretrained(str(instruction_adapter_dir))
tokenizer.save_pretrained(str(instruction_adapter_dir))

print('Instruction adapter saved to:', instruction_adapter_dir)
print('Saved files:', list(instruction_adapter_dir.iterdir()))

In [ ]:
# 24) Instruction-style inference helpers
import re
import torch

def build_instruction_prompt(instruction, input_text=''):
    instruction = instruction.strip()
    input_text = input_text.strip()

    if input_text:
        return (
            f'### Instruction:\n{instruction}\n\n'
            f'### Input:\n{input_text}\n\n'
            f'### Response:\n'
        )

    return (
        f'### Instruction:\n{instruction}\n\n'
        f'### Response:\n'
    )

def generate_instruction_response(instruction, input_text='', max_new_tokens=150):
    prompt = build_instruction_prompt(instruction, input_text)
    inputs = tokenizer(prompt, return_tensors='pt').to(instruction_model.device)

    with torch.no_grad():
        outputs = instruction_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=config.instruction_inference_do_sample,
            temperature=config.instruction_inference_temperature,
            top_p=config.instruction_inference_top_p,
            repetition_penalty=config.instruction_inference_repetition_penalty,
            no_repeat_ngram_size=config.instruction_inference_no_repeat_ngram_size,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer_text = generated_text[len(prompt):].strip()

    # Remove repeated header artifacts from generation output when present.
    answer_text = re.sub(r'\bInstruction\b(\s+Instruction\b)+', '', answer_text).strip()
    return answer_text

instruction_test_questions = [
    'What should I do if my bank charged an incorrect fee?',
    'How can I dispute an unauthorized charge on my credit card?',
    'What can I do if my payment was posted late because of a bank error?',
    'How do I request a refund for an unfair interest charge?',
    'What should I do if my account was charged fees after a system transition?',
]

for index, question in enumerate(instruction_test_questions, start=1):
    response = generate_instruction_response(question, max_new_tokens=150)
    print(f'Q{index}:', question)
    print(f'A{index}:', response)
    print('-' * 80)